<a href="https://colab.research.google.com/github/djibrillaseydouhamadou/Biodiversity-Species-Prediction/blob/main/TerraClimate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Terra Climate Sample Notebook                 

This notebooks demonstrates how to access the TerraClimate dataset and how to create a local GeoTIFF file. TerraClimate is a dataset of monthly climate and climatic water balance for global terrestrial surfaces from 1958 to the present. These data provide important inputs for ecological and hydrological studies at global scales that require high spatial resolution and time-varying data. All data have monthly temporal resolution and a ~4-km (1/24th degree) spatial resolution. This dataset is provided in Zarr format.

For more information, visit: https://planetarycomputer.microsoft.com/dataset/terraclimate#overview

## Load Python Dependencies

In [ ]:
# Install missing packages
!pip install rioxarray pystac-client planetary-computer zarr adlfs --quiet

In [ ]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')



# Import common GIS tools
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import rioxarray as rio
import rasterio
from matplotlib.cm import jet
import pandas as pd

# Import Planetary Computer tools
import pystac_client
import planetary_computer


from tqdm import tqdm

## Loading TerraClimate Data

In [ ]:
# Access STAC catalog and collection.
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace)

collection = catalog.get_collection("terraclimate")
asset = collection.assets["zarr-abfs"]

In [ ]:
# Open dataset and remove CRS.
ds = xr.open_dataset(asset.href,**asset.extra_fields["xarray:open_kwargs"])
ds = ds.drop('crs', dim=None) # Remove the CRS coordinate in the dataset
ds

A list of the available parameters is shown below:
![image.png](attachment:572022af-ed73-4ad2-bf78-e9db3b2bfc55.png)

In [ ]:
# Get the total memory usage in bytes
total_bytes = ds.nbytes

# Print the memory usage in a common format
print(f"Total memory usage: {total_bytes / 1024**2 / 1024:.0f} GB")

In [ ]:
# Since this is a HUGE dataset (nearly 2 TB), we should parse the dataset
# Trimming dataset to years 2017 thru 2019
ds = ds.sel(time=slice("2017-11-01", "2019-11-01"))

In [ ]:
# Trimming dataset to the desired Lat-Lon bounds (southeastern Australia)

min_lon = 139.94
min_lat = -39.74
max_lon = 151.48
max_lat = -30.92

mask_lon = (ds.lon >= min_lon) & (ds.lon <= max_lon)
mask_lat = (ds.lat >= min_lat) & (ds.lat <= max_lat)

In [ ]:
# Crop the dataset to smaller Lat-Lon regions
ds = ds.where(mask_lon & mask_lat, drop=True)
ds

In [ ]:
# Get the total memory usage in bytes
total_bytes2 = ds.nbytes

# Print the memory usage in a common format ... notice it is MUCH smaller now!
print(f"Total memory usage: {total_bytes2 / 1024**2:.0f} MB")

## Exploring the Data

In [ ]:
# Plot mean temperature over the region for 2 years
# This will demonstrate the expected seasonal variation
temperature = ds["tmax"].mean(dim=["lat", "lon"])
temperature.plot(figsize=(12, 6));

In [ ]:
# Plot monthly accumulated precipitation over the region for 2 years
temperature = ds["ppt"].mean(dim=["lat", "lon"])
temperature.plot(figsize=(12, 6));

In [ ]:
# Calculate the median of the dataset over time
soil_median = ds.soil.median(dim="time").compute()

In [ ]:
# Plot median soil moisture.
fig, ax = plt.subplots(figsize=(8,7))
soil_median.plot.imshow(vmin=0, vmax=40, cmap="jet")
plt.title("Median Soil Moisture")
plt.axis('off')
plt.show()

In [ ]:
# Persist dataset in memory.
ds=ds.persist()

In [ ]:
# Compute median along time dimension.
median = ds.median(dim="time", skipna=True).compute()

### Save the output data in a GeoTIFF file
The output product only contains 2 selected parameters that are used in the benchmark notebook. Participants may choose to include all of the parameters for their models to investigate how different parameters change their model results.

In [ ]:
#Define output file name.
filename = "TerraClimate_output.tiff"

In [ ]:
# Calculate the dimensions of the file
height = median.dims["lat"]
width = median.dims["lon"]

In [ ]:
# # Define the Coordinate Reference System (CRS) to be common Lat-Lon coordinates
# # Define the tranformation using our bounding box so the Lat-Lon information is written to the GeoTIFF
gt = rasterio.transform.from_bounds(min_lon,min_lat,max_lon,max_lat,width,height)
median.rio.write_crs("epsg:4326", inplace=True)
median.rio.write_transform(transform=gt, inplace=True);


In [ ]:
# Create the GeoTIFF output file using the defined parameters
with rasterio.open(filename,'w',driver='GTiff',width=width,height=height,
                   crs='epsg:4326',transform=gt,count=2,compress='lzw',dtype='float64') as dst:
    dst.write(median.srad,1)
    dst.write(median.vap,2)

In [ ]:
# Show the location and size of the new output file
!ls *.tiff

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# Charger tes données (adapte les noms de fichiers)
train = pd.read_csv('/content/Training_Data.csv')
test = pd.read_csv('/content/Test.csv')

# Séparer Target et Features
y = train['Occurrence Status']
train_df = train.drop(['Occurrence Status', 'ID'], axis=1, errors='ignore')
test_df = test.drop(['ID'], axis=1, errors='ignore')

In [ ]:
X_final = train_df.copy()
test_final = test_df.copy()

print("X_final and test_final created.")

In [ ]:
!pip install optuna geopandas xgboost lightgbm catboost -q


In [ ]:
# 1. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import optuna

import geopandas as gpd


In [ ]:
# Charger les fichiers
import pandas as pd

train = pd.read_csv('/content/Training_Data.csv')
test  = pd.read_csv('/content/Test.csv')

print("=== TRAIN ===")
print(train.shape)
print(train.head())
print(train.dtypes)


In [ ]:
print("=== TEST ===")
print(test.shape)
print(test.head())
print(test.dtypes)


In [ ]:
print(train.columns.tolist())
print(train.head(3))


In [ ]:
# Distribution de la cible
print("=== Distribution ===")
print(train["Occurrence Status"].value_counts())
print(train["Occurrence Status"].value_counts(normalize=True).mul(100).round(2))

print("\n=== Shape train / test ===")
print("Train:", train.shape)
print("Test:", test.shape)

print("\n=== Colonnes test ===")
print(test.columns.tolist())
print(test.head(3))


In [ ]:
!pip install planetary-computer pystac-client odc-stac rioxarray -q


In [ ]:
import planetary_computer
import pystac_client
import rioxarray
import numpy as np
import pandas as pd
from tqdm import tqdm

URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

catalog = pystac_client.Client.open(
    URL,
    modifier=planetary_computer.sign_inplace
)

VARIABLES = [
    "aet", "def", "pet", "ppt", "q", "soil",
    "srad", "swe", "tmax", "tmin", "vap", "ws", "vpd", "PDSI"
]

DATE_RANGE = "2017-11-01/2019-11-30"

print("Connexion OK")


In [ ]:
def extract_terraclimate_for_point(lat, lon, date_range=DATE_RANGE):
    bbox = [lon - 0.5, lat - 0.5, lon + 0.5, lat + 0.5]

    search = catalog.search(
        collections=["terraclimate"],
        bbox=bbox,
        datetime=date_range
    )
    items = list(search.get_items())

    if not items:
        return {f"{v}_{s}": np.nan for v in VARIABLES for s in ["mean", "std", "min", "max"]}

    features = {}
    for var in VARIABLES:
        values = []
        for item in items:
            if var in item.assets:
                try:
                    ds = rioxarray.open_rasterio(
                        planetary_computer.sign(item.assets[var].href),
                        masked=True
                    )
                    val = ds.sel(x=lon, y=lat, method="nearest").values.flatten()
                    val = val[~np.isnan(val)]
                    if len(val) > 0:
                        values.append(float(val[0]))
                except:
                    continue

        if values:
            features[f"{var}_mean"] = np.mean(values)
            features[f"{var}_std"]  = np.std(values)
            features[f"{var}_min"]  = np.min(values)
            features[f"{var}_max"]  = np.max(values)
        else:
            features[f"{var}_mean"] = np.nan
            features[f"{var}_std"]  = np.nan
            features[f"{var}_min"]  = np.nan
            features[f"{var}_max"]  = np.nan

    return features


In [ ]:
# Test rapide sur le premier point du train
row = train.iloc[0]
test_extraction = extract_terraclimate_for_point(row["Latitude"], row["Longitude"])
print(f"Nombre de features extraites : {len(test_extraction)}")
print(pd.Series(test_extraction).head(10))


In [ ]:
# Diagnostic 1 : voir ce que TerraClimate contient exactement
search_all = catalog.search(
    collections=["terraclimate"],
    datetime=DATE_RANGE,
    max_items=5
)
items_all = list(search_all.get_items())
print(f"Items sans filtre spatial : {len(items_all)}")

if items_all:
    item = items_all[0]
    print(f"Assets : {list(item.assets.keys())}")
    print(f"BBox de l'item : {item.bbox}")
    print(f"Date : {item.datetime}")


In [ ]:
# Diagnostic : lister toutes les collections disponibles
collections = list(catalog.get_collections())
for col in collections:
    print(col.id)


In [ ]:
# Étape 1 : Lister tous les fichiers disponibles dans l'environnement
import os

for f in os.listdir("/content"):
    print(f)


In [ ]:
# Étape 2 : Lire le GeoTIFF fourni et voir combien de bandes il contient
import rasterio

with rasterio.open("TerraClimate_output.tiff") as src:
    print(f"Nombre de bandes : {src.count}")
    print(f"Résolution : {src.res}")
    print(f"CRS : {src.crs}")
    print(f"Bounds : {src.bounds}")
    print(f"Shape : {src.shape}")


In [ ]:
# Étape 3 : Reproduire le pipeline de base pour avoir une baseline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
import joblib

# Charger les données
train = pd.read_csv("Training_Data.csv")
test  = pd.read_csv("Test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


In [ ]:
# Fonction d'extraction du GeoTIFF (identique au notebook officiel)
def map_satellite_data(tiff_path, df):
    with rasterio.open(tiff_path) as dataset:
        srad_data = dataset.read(1)
        vap_data  = dataset.read(2)

        lon = np.linspace(dataset.bounds.left, dataset.bounds.right, dataset.width)
        lat = np.linspace(dataset.bounds.top, dataset.bounds.bottom, dataset.height)

        srad_da = xr.DataArray(srad_data, coords=[("lat", lat), ("lon", lon)])
        vap_da  = xr.DataArray(vap_data,  coords=[("lat", lat), ("lon", lon)])

    srad_values, vap_values = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            srad_values.append(float(srad_da.sel(lat=row["Latitude"], lon=row["Longitude"], method="nearest").values))
            vap_values.append(float(vap_da.sel(lat=row["Latitude"],  lon=row["Longitude"], method="nearest").values))
        except:
            srad_values.append(np.nan)
            vap_values.append(np.nan)

    return pd.DataFrame({"srad": srad_values, "vap": vap_values})


In [ ]:
# Extraction train et test
train_feats = map_satellite_data("TerraClimate_output.tiff", train)
test_feats  = map_satellite_data("TerraClimate_output.tiff", test)

# Assemblage
train_df = pd.concat([train, train_feats], axis=1)
test_df  = pd.concat([test,  test_feats],  axis=1)

print(train_df.head(3))
print("NaN train:", train_df.isnull().sum().sum())


In [ ]:
!pip install stackstac -q


In [ ]:
import pystac_client
import planetary_computer

# On définit l'URL à l'extérieur
stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"

# On l'utilise directement dans Client.open()
catalog = pystac_client.Client.open(
    stac_url, # <-- Pas de "URL =" ici, juste la variable
    modifier=planetary_computer.sign_inplace
)

print("✅ Catalogue connecté !")

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# Chargement
train = pd.read_csv("Training_Data.csv")
test  = pd.read_csv("Test.csv")

# Extraction GeoTIFF
def map_satellite_data(tiff_path, df):
    with rasterio.open(tiff_path) as dataset:
        srad_data = dataset.read(1)
        vap_data  = dataset.read(2)
        lon = np.linspace(dataset.bounds.left,  dataset.bounds.right,  dataset.width)
        lat = np.linspace(dataset.bounds.top,   dataset.bounds.bottom, dataset.height)
        srad_da = xr.DataArray(srad_data, coords=[("lat", lat), ("lon", lon)])
        vap_da  = xr.DataArray(vap_data,  coords=[("lat", lat), ("lon", lon)])

    srad_values, vap_values = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            srad_values.append(float(srad_da.sel(lat=row["Latitude"], lon=row["Longitude"], method="nearest").values))
            vap_values.append(float(vap_da.sel(lat=row["Latitude"],   lon=row["Longitude"], method="nearest").values))
        except:
            srad_values.append(np.nan)
            vap_values.append(np.nan)

    return pd.DataFrame({"srad": srad_values, "vap": vap_values})

train_feats = map_satellite_data("TerraClimate_output.tiff", train)
test_feats  = map_satellite_data("TerraClimate_output.tiff", test)

train_df = pd.concat([train, train_feats], axis=1)
test_df  = pd.concat([test,  test_feats],  axis=1)

print("NaN train:", train_df.isnull().sum().sum())
print(train_df.head(3))


In [ ]:
features = ["srad", "vap"]
X = train_df[features].fillna(train_df[features].median())
y = train_df["Occurrence Status"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Logistic Regression (modèle officiel)
pipe_lr = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scl", MinMaxScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])
pipe_lr.fit(X_train, y_train)
print("F1 Logistic Regression :", f1_score(y_val, pipe_lr.predict(X_val)))

# Random Forest
rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
print("F1 Random Forest       :", f1_score(y_val, rf.predict(X_val)))


In [ ]:
search = catalog.search(
    collections=["terraclimate"],
    max_items=5
)
items = list(search.get_items())
print(f"Items trouvés : {len(items)}")

if items:
    item = items[0]
    print(f"Assets disponibles : {list(item.assets.keys())}")
    print(f"BBox : {item.bbox}")
    print(f"Date : {item.datetime}")


In [ ]:
from xgboost import XGBClassifier

# Paramètres optimisés pour ce challenge
model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1
)

model.fit(X_train, y_train)

In [ ]:
X_test = test_df[features].fillna(train_df[features].median())

print("X_test DataFrame created with missing values imputed.")
print(X_test.head())

In [ ]:
probs = model.predict_proba(X_test)[:, 1]
final_predictions = pd.Series(probs).rank(pct=True)

print("Probabilities predicted and ranked.")
print(final_predictions.head())

In [ ]:
%who DataFrame

In [ ]:
# 1. On fusionne les données météo (test_feats) avec le dataframe principal (test_df)
# On vérifie si les index correspondent
test_combined = pd.concat([test_df, test_feats], axis=1)

# 2. Maintenant on peut créer les variables de "Mémoire" (Lags)
# Si tu n'as qu'une ligne par ID, on ne peut pas faire de rolling sum classique.
# Mais si tu as extrait plusieurs mois, voici le code corrigé :

try:
    # On calcule le cumul de pluie (si ppt existe enfin)
    if 'ppt' in test_combined.columns:
        test_combined['ppt_3month_sum'] = test_combined.groupby('ID')['ppt'].transform(lambda x: x.rolling(window=3).sum())
    else:
        # Si ppt manque encore, on utilise ce qu'on a (vap, srad) pour créer des interactions
        test_combined['vap_srad_inter'] = test_combined['vap'] * test_combined['srad']
        print("⚠️ 'ppt' manque toujours, création d'une variable d'interaction à la place.")

    # 3. On prépare les données pour le modèle
    # On ne garde que les colonnes numériques qui servent de prédicteurs
    features_to_use = ['srad', 'vap', 'vap_srad_inter'] # Ajoute ppt_3month_sum si ça a marché
    X_test_final = test_combined[features_to_use].fillna(0)

    print("✅ Données prêtes pour le boosting !")

except Exception as e:
    print(f"❌ Erreur lors du calcul : {e}")
    print("Colonnes disponibles :", test_combined.columns.tolist())

In [ ]:
# 1. Nettoyage des doublons de colonnes
# On ne garde qu'une seule occurrence de chaque colonne
test_combined = test_combined.loc[:, ~test_combined.columns.duplicated()]

# 2. Création de variables d'interaction (Feature Engineering)
# Puisqu'on n'a pas encore 'ppt', on combine ce qu'on a pour créer du "signal"
try:
    # L'interaction Rayonnement x Vapeur aide à identifier les zones humides/chaudes
    test_combined['vap_srad_inter'] = test_combined['vap'] * test_combined['srad']

    # Si par chance tu as extrait d'autres colonnes entre temps
    if 'ppt' in test_combined.columns:
        test_combined['humidity_index'] = test_combined['ppt'] / (test_combined['vap'] + 1)

    print("✅ Variables d'interaction créées avec succès !")
except Exception as e:
    print(f"❌ Erreur persistante : {e}")

# 3. Sélection finale des variables pour le modèle
# On utilise uniquement les colonnes numériques
features_model = ['srad', 'vap', 'vap_srad_inter']
X_test_final = test_combined[features_model].fillna(test_combined[features_model].mean())

print("🚀 Prêt pour la prédiction. Colonnes utilisées :", features_model)

In [ ]:
import pandas as pd
import planetary_computer as pc
from pystac_client import Client
import numpy as np

# 1. Configuration
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=pc.sign_inplace)

def extract_climate_data(df):
    results = []
    print(f"Extraction de {len(df)} points en cours...")

    for index, row in df.iterrows():
        try:
            # Recherche de la zone
            search = catalog.search(
                collections=["terraclimate"],
                bbox=[row.Longitude - 0.1, row.Latitude - 0.1, row.Longitude + 0.1, row.Latitude + 0.1],
                datetime="2018-01-01/2018-12-31"
            )
            items = list(search.get_items())

            # Extraction des valeurs (moyenne annuelle pour simplifier et stabiliser)
            # On se concentre sur ppt (pluie) et pdsi (sécheresse)
            if items:
                # On utilise la méthode de Claude mais sécurisée
                asset_href = items[0].assets["netcdf"].href
                ds = xr.open_dataset(pc.sign(asset_href), decode_times=True)
                point_data = ds.sel(lat=row.Latitude, lon=row.Longitude, method='nearest').mean(dim='time')

                results.append({
                    'ppt': float(point_data.ppt.values),
                    'pdsi': float(point_data.pdsi.values),
                    'tmax': float(point_data.tmax.values)
                })
            if index % 50 == 0: print(f"Avancement : {index}/{len(df)}")
        except:
            results.append({'ppt': np.nan, 'pdsi': np.nan, 'tmax': np.nan})

    return pd.DataFrame(results)

# 2. Application sur test_df
test_extra_data = extract_climate_data(test_df)

# 3. Fusion finale
test_ready = pd.concat([test_df.reset_index(drop=True), test_extra_data], axis=1)
print("✅ Terminé ! Colonnes :", test_ready.columns.tolist())

In [ ]:
# 1. Extraction des données pour le TRAIN
print("Extraction des données d'entraînement en cours...")
train_extra_data = extract_climate_data(train_df)

# 2. Fusion pour créer train_ready
train_ready = pd.concat([train_df.reset_index(drop=True), train_extra_data], axis=1)

# 3. Vérification des colonnes
print("✅ train_ready est prêt ! Colonnes :", train_ready.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBClassifier

print("🌍 2. Recréation des Éco-régions (Feature Engineering Spatial)...")
kmeans = KMeans(n_clusters=15, random_state=42, n_init=10)

# CORRECTION ICI : On assigne d'abord, on convertit ensuite !
train_ready['eco_region'] = kmeans.fit_predict(train_ready[['Latitude', 'Longitude']])
train_ready['eco_region'] = train_ready['eco_region'].astype('category')

test_ready['eco_region'] = kmeans.predict(test_ready[['Latitude', 'Longitude']])
test_ready['eco_region'] = test_ready['eco_region'].astype('category')

print("🐸 3. Recréation de la Distance Spatiale Anti-Triche...")
coords_grenouilles = train_ready[train_ready['Occurrence Status'] == 1][['Latitude', 'Longitude']]
coords_grenouilles_rad = np.radians(coords_grenouilles)
train_rad = np.radians(train_ready[['Latitude', 'Longitude']])
test_rad = np.radians(test_ready[['Latitude', 'Longitude']])

# Entraînement de la distance sur le Train
knn_train = NearestNeighbors(n_neighbors=2, metric='haversine')
knn_train.fit(coords_grenouilles_rad)
distances_train = knn_train.kneighbors(train_rad)[0]

# Application anti-triche
train_ready['dist_grenouille_km'] = np.where(
    train_ready['Occurrence Status'] == 1,
    distances_train[:, 1],
    distances_train[:, 0]
) * 6371

# Distance pour le Test
knn_test = NearestNeighbors(n_neighbors=1, metric='haversine')
knn_test.fit(coords_grenouilles_rad)
test_ready['dist_grenouille_km'] = knn_test.kneighbors(test_rad)[0].flatten() * 6371

print("✅ Données prêtes ! Lancement de l'entraînement XGBoost...")

# -----------------------------------------
# 4. ENTRAÎNEMENT DU MODÈLE ULTIME
# -----------------------------------------
features_finales = ['srad', 'vap', 'ppt', 'pdsi', 'tmax', 'eco_region', 'dist_grenouille_km']

X_train_final = train_ready[features_finales]
y_train_final = train_ready['Occurrence Status']
X_test_final = test_ready[features_finales]

ratio_poids = (y_train_final == 0).sum() / (y_train_final == 1).sum()

model_xgb_ultime = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.015,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio_poids,
    enable_categorical=True,
    random_state=42
)

model_xgb_ultime.fit(X_train_final, y_train_final)

importance = pd.Series(model_xgb_ultime.feature_importances_, index=features_finales).sort_values(ascending=False)
print("\n📊 Importance des variables :")
print(importance)

predictions_ultimes = model_xgb_ultime.predict(X_test_final)

submission_finale = pd.DataFrame({
    'id': test_ready['ID'],
    'target': predictions_ultimes.astype(int)
})

nom_fichier = "SUBMISSION_NASA_097.csv"
submission_finale.to_csv(nom_fichier, index=False)

print(f"🎯 Fichier '{nom_fichier}' généré avec succès !")

In [ ]:
import requests
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

# ==========================================
# 1. FONCTION D'EXTRACTION DE L'ALTITUDE (EN LOTS)
# ==========================================
def get_elevation_fast(df):
    print(f"⛰️ Récupération de l'altitude pour {len(df)} points...")
    elevations = []
    batch_size = 100 # On demande 100 coordonnées par requête web !

    for i in range(0, len(df), batch_size):
        batch = df.iloc[i:i+batch_size]
        # On regroupe les latitudes et longitudes
        lats = ",".join(batch['Latitude'].round(4).astype(str))
        lons = ",".join(batch['Longitude'].round(4).astype(str))

        url = f"https://api.open-meteo.com/v1/elevation?latitude={lats}&longitude={lons}"
        try:
            resp = requests.get(url, timeout=10)
            if resp.status_code == 200:
                elevations.extend(resp.json()['elevation'])
            else:
                elevations.extend([np.nan] * len(batch))
        except:
            elevations.extend([np.nan] * len(batch))

    return elevations

# ==========================================
# 2. APPLICATION AUX DONNÉES
# ==========================================
print("🔄 Téléchargement en cours...")
train_ready['elevation'] = get_elevation_fast(train_ready)
test_ready['elevation'] = get_elevation_fast(test_ready)

# Sécurité : on remplace les potentiels ratés par la moyenne
train_ready['elevation'].fillna(train_ready['elevation'].mean(), inplace=True)
test_ready['elevation'].fillna(train_ready['elevation'].mean(), inplace=True)

print("✅ Altitudes intégrées avec succès !")

# ==========================================
# 3. LE MODÈLE XGBOOST 3D (Ton meilleur modèle + Altitude)
# ==========================================
# On ajoute l'élévation à notre liste de super-variables
features_3D = [
    'srad', 'vap', 'ppt', 'pdsi', 'tmax',
    'eco_region', 'dist_grenouille_km',
    'eau_dispo', 'stress_climat', 'elevation'  # 👈 La nouvelle venue
]

X_train_3D = train_ready[features_3D]
X_test_3D = test_ready[features_3D]
y_train_3D = train_ready['Occurrence Status']

ratio_poids = (y_train_3D == 0).sum() / (y_train_3D == 1).sum()

print("⚡ Entraînement du modèle Ultime 3D...")
model_xgb_3D = XGBClassifier(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.015,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio_poids,
    enable_categorical=True,
    random_state=42
)

model_xgb_3D.fit(X_train_3D, y_train_3D)

# Vérification du poids de l'altitude
importance = pd.Series(model_xgb_3D.feature_importances_, index=features_3D).sort_values(ascending=False)
print("\n📊 Importance des variables (Avec Altitude) :")
print(importance)

# ==========================================
# 4. SOUMISSION FINALE
# ==========================================
predictions_3D = model_xgb_3D.predict(X_test_3D)

submission_3D = pd.DataFrame({
    'id': test_ready['ID'],
    'target': predictions_3D.astype(int)
})

nom_fichier = "SUBMISSION_3D_097.csv"
submission_3D.to_csv(nom_fichier, index=False)

print(f"🎯 Fichier '{nom_fichier}' prêt ! En route pour briser le plafond de verre.")

In [ ]:
import pandas as pd
import numpy as np

def reduce_mem_usage(df):
    """
    Itère sur toutes les colonnes d'un dataframe et modifie le type de données
    pour réduire l'utilisation de la mémoire.
    """
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Utilisation mémoire initiale : {start_mem:.2f} MB')

    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object and not pd.api.types.is_categorical_dtype(df[col]):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Utilisation mémoire après optimisation : {end_mem:.2f} MB')
    print(f'Réduction de : {100 * (start_mem - end_mem) / start_mem:.1f}%')
    return df

def engineer_terraclimate_features(df, time_col='date', entity_col='location_id'):
    """
    Génère des caractéristiques climatiques avancées (deltas, amplitudes, cumuls).
    Suppose que le DataFrame a été trié chronologiquement.
    """
    # 1. Tri obligatoire pour les calculs de fenêtres glissantes
    df = df.sort_values(by=[entity_col, time_col]).reset_index(drop=True)

    # 2. Amplitudes thermiques (Stress thermique)
    if 'tmax' in df.columns and 'tmin' in df.columns:
        df['temp_amplitude'] = df['tmax'] - df['tmin']

    # Variables cibles pour les calculs de fenêtres et de variations
    vars_to_roll = ['pr', 'soil', 'tmax', 'tmin', 'aet'] # Ajustez selon les colonnes TerraClimate exactes
    existing_vars = [v for v in vars_to_roll if v in df.columns]

    # GroupBy sur l'identifiant géographique pour ne pas mélanger les données de lieux différents
    grouped = df.groupby(entity_col)

    # 3. Calcul des Deltas (Chocs climatiques d'un mois sur l'autre)
    for var in existing_vars:
        # Différence brute avec le mois précédent
        df[f'{var}_delta_1m'] = grouped[var].diff(periods=1)
        # Taux de changement (en pourcentage) pour normaliser l'impact
        # On ajoute un petit epsilon (1e-5) pour éviter les divisions par zéro
        df[f'{var}_pct_change_1m'] = grouped[var].pct_change(periods=1).fillna(0)

    # 4. Fenêtres glissantes (Rolling Windows : Cumuls et Moyennes)
    windows = [3, 6] # 3 mois (saison court terme), 6 mois (tendance moyen terme)

    for window in windows:
        for var in existing_vars:
            if var == 'pr': # Pour les précipitations, la somme (cumul) est souvent plus pertinente
                df[f'{var}_roll_sum_{window}m'] = grouped[var].transform(lambda x: x.rolling(window, min_periods=1).sum())

            # Pour toutes les variables, la moyenne lissée
            df[f'{var}_roll_mean_{window}m'] = grouped[var].transform(lambda x: x.rolling(window, min_periods=1).mean())

            # La variance (capture l'instabilité du climat sur la période)
            df[f'{var}_roll_std_{window}m'] = grouped[var].transform(lambda x: x.rolling(window, min_periods=2).std())

    # 5. Nettoyage des valeurs NaN générées par les décalages (diff/rolling)
    # Remplacer par 0 ou par la moyenne globale de la colonne selon la stratégie choisie
    df = df.fillna(0)

    return df

# ==========================================
# Exemple d'exécution du pipeline
# ==========================================

# 1. Charger vos données (en omettant lat/lon comme prédicteurs)
# df_train = pd.read_csv('votre_fichier_train.csv')

# 2. Optimiser la mémoire immédiatement après le chargement
# df_train = reduce_mem_usage(df_train)

# 3. Appliquer l'ingénierie des caractéristiques
# df_train_engineered = engineer_terraclimate_features(df_train, time_col='date_observation', entity_col='id_observation')

# 4. Re-optimiser une dernière fois car de nouvelles colonnes ont été créées
# df_train_final = reduce_mem_usage(df_train_engineered)

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

# 1. Préparation des données avec la nouvelle variable
target_col = 'Occurrence Status'
cols_to_drop = [target_col, 'ID', 'Latitude', 'Longitude']
# On inclut explicitement climate_cluster s'il existe
features = [c for c in train_ready.columns if c not in cols_to_drop]

X = train_ready[features]
y = train_ready[target_col]

# 2. Configuration de LightGBM
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'is_unbalance': True,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

# 3. Validation croisée
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
feature_importances = np.zeros(len(features))
oof_preds = np.zeros(len(X))

print(f"Réentraînement sur {len(features)} variables (incluant climate_cluster)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

    model = lgb.train(
        lgb_params,
        train_data,
        valid_sets=[train_data, val_data],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    val_preds = model.predict(X_val, num_iteration=model.best_iteration)
    oof_preds[val_idx] = val_preds
    feature_importances += model.feature_importance(importance_type='gain') / skf.n_splits

# 4. Évaluation
oof_preds_binary = (oof_preds > 0.5).astype(int)
print(f"\nNouveau Score F1 global (OOF) : {f1_score(y, oof_preds_binary):.4f}")

# 5. Visualisation de l'importance
importance_df = pd.DataFrame({'Feature': features, 'Value': feature_importances}).sort_values(by='Value', ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(x='Value', y='Feature', data=importance_df, palette='magma')
plt.title('Importance des variables avec Climate Cluster')
plt.show()

In [ ]:
`from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

def apply_climate_clustering(df, n_clusters=15):
    """
    Cre une variable de cluster base uniquement sur les caractristiques climatiques.
    """
    print(f"Lancement du clustering sur {n_clusters} zones climatiques...")

    # 1. Slection des variables climatiques de base
    climate_vars = ['pr', 'soil', 'tmax', 'tmin', 'aet', 'vap', 'srad']
    existing_vars = [v for v in climate_vars if v in df.columns]

    # 2. Normalisation
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df[existing_vars].fillna(0))

    # 3. Clustering
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        batch_size=1024,
        random_state=42,
        n_init=3
    )

    # On cre une nouvelle colonne 'climate_cluster'
    df['climate_cluster'] = kmeans.fit_predict(scaled_data)

    # Conversion en catgorie pour LightGBM
    df['climate_cluster'] = df['climate_cluster'].astype('category')

    print("Clustering termin et intgr au DataFrame.")
    return df

# ==========================================
# Excution et Analyse (CORRIGE)
# ==========================================

# Utilisation de train_ready (variable existante)
train_ready = apply_climate_clustering(train_ready, n_clusters=15)

# Analyse de pertinence : utilise 'Occurrence Status' comme cible
cluster_analysis = train_ready.groupby('climate_cluster')['Occurrence Status'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
cluster_analysis.plot(kind='bar', color='skyblue')
plt.title('Taux de prsence des grenouilles par Cluster Climatique')
plt.ylabel('Moyenne de la cible')
plt.xlabel('Cluster ID')
plt.axhline(train_ready['Occurrence Status'].mean(), color='red', linestyle='--', label='Moyenne globale')
plt.legend()
plt.show()

In [ ]:
# Application du clustering au jeu de TEST
# On utilise la même fonction que pour le train pour rester cohérent
print("Application du clustering climatique au jeu de test...")
test_ready = apply_climate_clustering(test_ready, n_clusters=15)

# Vérification que la colonne est bien présente
if 'climate_cluster' in test_ready.columns:
    print("✅ Colonne 'climate_cluster' ajoutée au jeu de test.")
else:
    print("❌ Erreur lors de l'ajout de la colonne.")

In [ ]:
from sklearn.metrics import f1_score
import numpy as np
import matplotlib.pyplot as plt

def optimize_f1_threshold(y_true, y_probs):
    """
    Teste une plage de seuils pour trouver celui qui maximise le score F1.
    """
    thresholds = np.arange(0.1, 0.9, 0.01)
    f1_scores = []

    print("Recherche du seuil optimal en cours...")

    for thresh in thresholds:
        # Convertir les probabilités en classes binaires selon le seuil testé
        preds_binary = (y_probs > thresh).astype(int)
        score = f1_score(y_true, preds_binary)
        f1_scores.append(score)

    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    print(f"--- Résultat de l'optimisation ---")
    print(f"Seuil par défaut (0.50) F1 : {f1_score(y_true, (y_probs > 0.5).astype(int)):.4f}")
    print(f"Nouveau Seuil Optimal ({best_threshold:.2f}) F1 : {best_f1:.4f}")
    print(f"Gain estimé : {best_f1 - f1_score(y_true, (y_probs > 0.5).astype(int)):.4f}")

    # Visualisation de la courbe
    plt.figure(figsize=(8, 5))
    plt.plot(thresholds, f1_scores, label='F1 Score', color='teal', linewidth=2)
    plt.axvline(best_threshold, color='red', linestyle='--', label=f'Seuil Opt. ({best_threshold:.2f})')
    plt.title("Optimisation du Seuil de Décision pour le F1-Score")
    plt.xlabel("Seuil de probabilité")
    plt.ylabel("F1-Score")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    return best_threshold

# ==========================================
# Exécution sur vos prédictions LightGBM
# (Suppose que oof_preds contient les probabilités générées par votre modèle)
# ==========================================

# optimal_thresh = optimize_f1_threshold(y, oof_preds)

# Lors de votre prédiction finale sur le jeu de test pour soumission :
# test_preds_probs = model.predict(X_test)
# final_submission_classes = (test_preds_probs > optimal_thresh).astype(int)

In [ ]:
import pandas as pd

# --- Nettoyage et Prédiction ---
# On utilise 'test_ready' et le modèle LightGBM entraîné

# Seuil optimisé (ajusté selon l'analyse précédente)
optimal_thresh = 0.42

print(f"Génération des prédictions avec le seuil de {optimal_thresh}...")

# 1. Prédiction des probabilités
test_preds_probs = model.predict(test_ready[features])

# 2. Conversion binaire via le seuil
final_predictions = (test_preds_probs > optimal_thresh).astype(int)

# 3. Création du DataFrame au format Zindi
submission = pd.DataFrame({
    'ID': test_ready['ID'],
    'Occurrence Status': final_predictions
})

# 4. Export CSV
nom_fichier = 'soumission_lightgbm_final.csv'
submission.to_csv(nom_fichier, index=False)

print(f"Succès ! Le fichier '{nom_fichier}' a été généré.")
print(f"Distribution des classes :")
print(submission['Occurrence Status'].value_counts(normalize=True) * 100)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

def kfold_target_encoding(train, test, cat_cols, target_col, n_splits=5, smoothing=10, noise_level=0.01):
    """
    Applique un Target Encoding régularisé.
    Correction : On convertit explicitement la colonne en type objet/numérique pendant le mapping
    pour éviter les restrictions du type Categorical.
    """
    train_encoded = train.copy()
    test_encoded = test.copy()
    global_mean = train[target_col].mean()
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    for col in cat_cols:
        encoded_values = np.zeros(len(train))

        # 1. K-Fold Encoding
        for train_idx, val_idx in skf.split(train, train[target_col]):
            X_tr = train.iloc[train_idx]
            X_val = train.iloc[val_idx]

            # Calcul des statistiques
            stats = X_tr.groupby(col)[target_col].agg(['count', 'mean'])
            smoothed = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)

            # On convertit la colonne de validation en objet pour accepter des floats
            mapped = X_val[col].astype(object).map(smoothed)
            encoded_values[val_idx] = mapped.fillna(global_mean).values

        # Ajout du bruit et assignation
        noise = np.random.normal(0, noise_level, len(train))
        train_encoded[f'{col}_target_enc'] = encoded_values + noise

        # 2. Encoding sur le jeu de test
        stats_full = train.groupby(col)[target_col].agg(['count', 'mean'])
        smoothed_full = (stats_full['count'] * stats_full['mean'] + smoothing * global_mean) / (stats_full['count'] + smoothing)

        test_encoded[f'{col}_target_enc'] = test[col].astype(object).map(smoothed_full).fillna(global_mean)

    return train_encoded, test_encoded

# ==========================================
# Exécution du Target Encoding
# ==========================================

colonnes_a_encoder = ['climate_cluster']

train_ready_enc, test_ready_enc = kfold_target_encoding(
    train=train_ready,
    test=test_ready,
    cat_cols=colonnes_a_encoder,
    target_col='Occurrence Status',
    n_splits=5,
    smoothing=20,
    noise_level=0.005
)

print("✅ Target Encoding terminé avec succès.")
print(f"Nouvelle colonne ajoutée : {train_ready_enc.columns[-1]}")

In [ ]:
import catboost as cb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

# 1. Définition des caractéristiques (Features)
target_col = 'Occurrence Status'
cols_to_drop = [target_col, 'ID', 'Latitude', 'Longitude', 'climate_cluster']

features = [c for c in train_ready_enc.columns if c not in cols_to_drop]

# Identification automatique des variables catégorielles (comme eco_region)
cat_features = [f for f in features if train_ready_enc[f].dtype.name == 'category']

X = train_ready_enc[features]
y = train_ready_enc[target_col]

# 2. Paramètres CatBoost
cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'F1',
    'learning_rate': 0.05,
    'iterations': 1000,
    'depth': 6,
    'auto_class_weights': 'Balanced',
    'random_seed': 42,
    'verbose': 100,
    'od_type': 'Iter',
    'od_wait': 50,
    'cat_features': cat_features # <-- Ajout crucial ici
}

# 3. Validation Croisée (5 Folds)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cb_oof_preds = np.zeros(len(X))
cb_test_preds = np.zeros(len(test_ready_enc))

print(f"Entraînement sur {len(features)} variables (Catégories: {cat_features})...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    model_cb = cb.CatBoostClassifier(**cb_params)

    model_cb.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        use_best_model=True,
        plot=False
    )

    cb_oof_preds[val_idx] = model_cb.predict_proba(X_val)[:, 1]
    cb_test_preds += model_cb.predict_proba(test_ready_enc[features])[:, 1] / skf.n_splits

# 4. Évaluation OOF
cb_oof_binary = (cb_oof_preds > 0.5).astype(int)
print(f"\nScore F1 CatBoost (OOF) : {f1_score(y, cb_oof_binary):.4f}")

In [ ]:
# 1. Ensemble: Weighted average of CatBoost and LightGBM probabilities
# Using the OOF probabilities stored previously (oof_preds for LGBM, cb_oof_preds for CatBoost)

# Let's try a simple 50/50 blend first
ensemble_oof_probs = (oof_preds + cb_oof_preds) / 2
ensemble_test_probs = (test_preds_probs + cb_test_preds) / 2

# 2. Re-optimize threshold for the ensemble
def find_best_threshold(y_true, y_probs):
    thresholds = np.arange(0.1, 0.9, 0.01)
    scores = [f1_score(y_true, (y_probs > t).astype(int)) for t in thresholds]
    return thresholds[np.argmax(scores)], np.max(scores)

best_thresh, best_f1 = find_best_threshold(y, ensemble_oof_probs)

print(f"--- Ensemble Results ---")
print(f"Optimal Threshold: {best_thresh:.2f}")
print(f"Ensemble OOF F1-Score: {best_f1:.4f}")

# 3. Create Final Submission
final_sub_preds = (ensemble_test_probs > best_thresh).astype(int)

submission_ensemble = pd.DataFrame({
    'ID': test_ready['ID'],
    'Occurrence Status': final_sub_preds
})

filename = 'submission_ensemble_cat_lgb.csv'
submission_ensemble.to_csv(filename, index=False)

print(f"\n✅ Final ensemble submission saved as: {filename}")
print("Distribution of classes:")
print(submission_ensemble['Occurrence Status'].value_counts(normalize=True) * 100)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

# ==========================================
# Validation Adversarielle (Version Corrigée)
# ==========================================

print("Préparation des données pour la Validation Adversarielle...")

# 1. Utilisation des bonnes variables : train_ready_enc et test_ready_enc
train_adv = train_ready_enc.copy()
test_adv = test_ready_enc.copy()

train_adv['is_test'] = 0
test_adv['is_test'] = 1

# 2. Concaténation des deux jeux
adv_data = pd.concat([train_adv, test_adv], axis=0, ignore_index=True)

# 3. Définition des variables explicatives
# On retire la cible réelle, l'ID, et les colonnes de calcul technique
# On garde les features climatiques pour voir si elles diffèrent entre Train et Test
cols_to_drop_adv = ['is_test', 'ID', 'Occurrence Status', 'Latitude', 'Longitude']

features_adv = [c for c in adv_data.columns if c not in cols_to_drop_adv]

X_adv = adv_data[features_adv]
y_adv = adv_data['is_test']

# 4. Configuration LightGBM
adv_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

# 5. Validation Croisée
skf_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
auc_scores = []
feature_importances_adv = np.zeros(len(features_adv))

print(f"Entraînement du modèle sur {len(features_adv)} variables...")

for train_idx, val_idx in skf_adv.split(X_adv, y_adv):
    X_tr, y_tr = X_adv.iloc[train_idx], y_adv.iloc[train_idx]
    X_va, y_va = X_adv.iloc[val_idx], y_adv.iloc[val_idx]

    train_data_adv = lgb.Dataset(X_tr, label=y_tr)
    val_data_adv = lgb.Dataset(X_va, label=y_va, reference=train_data_adv)

    model_adv = lgb.train(
        adv_params,
        train_data_adv,
        valid_sets=[train_data_adv, val_data_adv],
        num_boost_round=500,
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )

    preds_adv = model_adv.predict(X_va, num_iteration=model_adv.best_iteration)
    auc_scores.append(roc_auc_score(y_va, preds_adv))
    feature_importances_adv += model_adv.feature_importance(importance_type='gain') / skf_adv.n_splits

mean_auc = np.mean(auc_scores)
print(f"\nScore AUC Adversariel : {mean_auc:.4f}")

# 6. Interprétation
if mean_auc > 0.70:
    print("\n⚠️ ALERTE : Le modèle arrive à trop bien distinguer le Train du Test.")
else:
    print("\n✅ OK : Les distributions sont cohérentes entre Train et Test.")

importance_df_adv = pd.DataFrame({'Feature': features_adv, 'Value': feature_importances_adv})
importance_df_adv = importance_df_adv.sort_values(by='Value', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Value', y='Feature', data=importance_df_adv, palette='Reds_r')
plt.title('Variables distinguant le Train du Test')
plt.show()

In [ ]:
import catboost as cb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# 1. Mise à jour des colonnes à exclure (On élimine les "traîtres" identifiés par la validation adversarielle)
cols_to_drop_final = [
    'Occurrence Status',
    'ID',
    'Latitude',
    'Longitude',
    'dist_grenouille_km',
    'eco_region',
    'elevation',
    'climate_cluster_target_enc', # <-- Suppression du Target Encoding toxique (cause du shift)
    'srad'                        # <-- Suppression de la variable climatique la plus instable
]

# On utilise les variables existantes dans le kernel : train_ready_enc et test_ready_enc
# On garde 'climate_cluster' en tant que catégorie simple
features_final = [c for c in train_ready_enc.columns if c not in cols_to_drop_final]

X_train_clean = train_ready_enc[features_final]
y_train_clean = train_ready_enc['Occurrence Status']
X_test_clean = test_ready_enc[features_final]

print(f"Entraînement avec {len(features_final)} variables (sans Target Encoding ni srad)... balance: {y_train_clean.mean():.2f}")

# 2. Entraînement LightGBM et CatBoost (Ensemble)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_test_preds = np.zeros(len(X_test_clean))
cb_test_preds = np.zeros(len(X_test_clean))

# CatBoost gère les catégories nativement
cat_features_indices = [X_train_clean.columns.get_loc('climate_cluster')] if 'climate_cluster' in X_train_clean.columns else []

for train_idx, val_idx in skf.split(X_train_clean, y_train_clean):
    X_tr, y_tr = X_train_clean.iloc[train_idx], y_train_clean.iloc[train_idx]
    X_va, y_va = X_train_clean.iloc[val_idx], y_train_clean.iloc[val_idx]

    # --- LightGBM ---
    lgb_model = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=31, is_unbalance=True, random_state=42, verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_test_preds += lgb_model.predict_proba(X_test_clean)[:, 1] / 5

    # --- CatBoost ---
    cb_model = cb.CatBoostClassifier(learning_rate=0.05, depth=6, auto_class_weights='Balanced', random_seed=42, verbose=0)
    cb_model.fit(X_tr, y_tr, cat_features=cat_features_indices, eval_set=(X_va, y_va), early_stopping_rounds=50)
    cb_test_preds += cb_model.predict_proba(X_test_clean)[:, 1] / 5

# 3. Moyenne des prédictions (Ensemble simple 50/50)
final_ensemble_probs = (lgb_test_preds + cb_test_preds) / 2

# 4. Génération de la soumission avec un seuil conservateur (0.50)
final_preds_binary = (final_ensemble_probs > 0.50).astype(int)

submission = pd.DataFrame({
    'ID': test_ready_enc['ID'],
    'Occurrence Status': final_preds_binary
})
submission.to_csv('soumission_robuste_sans_shift.csv', index=False)
print("Fichier 'soumission_robuste_sans_shift.csv' généré avec succès.")

In [ ]:
import pandas as pd
import numpy as np

# ==========================================
# Pseudo-Étiquetage basé sur l'Ensemble
# ==========================================

print("Démarrage du Pseudo-Étiquetage avec les probabilités de l'Ensemble...")

# 1. Définition des seuils de confiance très stricts
# On ne prend aucun risque : on ne veut que les certitudes absolues
seuil_presence = 0.95
seuil_absence = 0.05

# 2. Copie du jeu de test propre
X_test_pseudo = X_test_clean.copy()
X_test_pseudo['prob_ensemble'] = final_ensemble_probs

# 3. Extraction des pseudo-étiquettes
pseudo_positives = X_test_pseudo[X_test_pseudo['prob_ensemble'] >= seuil_presence].copy()
pseudo_positives['target'] = 1

pseudo_negatives = X_test_pseudo[X_test_pseudo['prob_ensemble'] <= seuil_absence].copy()
pseudo_negatives['target'] = 0

# Suppression de la colonne de probabilité pour correspondre au format d'entraînement
pseudo_positives = pseudo_positives.drop(columns=['prob_ensemble'])
pseudo_negatives = pseudo_negatives.drop(columns=['prob_ensemble'])

print(f"Nouvelles observations 'Présence' (haute confiance) ajoutées : {len(pseudo_positives)}")
print(f"Nouvelles observations 'Absence' (haute confiance) ajoutées : {len(pseudo_negatives)}")

# 4. Fusion avec le jeu d'entraînement d'origine
# On reconstitue d'abord X_train_clean et y_train_clean en un seul DataFrame
df_train_temp = X_train_clean.copy()
df_train_temp['target'] = y_train_clean

# Concaténation globale
df_train_boosted = pd.concat([df_train_temp, pseudo_positives, pseudo_negatives], ignore_index=True)

# 5. Séparation des nouvelles données boostées
X_train_boosted = df_train_boosted.drop(columns=['target'])
y_train_boosted = df_train_boosted['target']

print(f"Taille du jeu d'entraînement d'origine : {len(df_train_temp)}")
print(f"Nouvelle taille du jeu d'entraînement boosté : {len(df_train_boosted)}")

In [ ]:
import numpy as np
import pandas as pd

def generate_safe_interactions(df, base_features):
    """
    Crée des interactions mathématiques non-linéaires uniquement
    à partir des variables pures validées.
    """
    df_inter = df.copy()

    # Assurez-vous que ces variables existent dans vos 5 rescapées.
    # Ajustez les noms selon votre dataframe exact.
    if 'tmax' in base_features and 'tmin' in base_features:
        df_inter['temp_range'] = df_inter['tmax'] - df_inter['tmin']
        df_inter['temp_mean'] = (df_inter['tmax'] + df_inter['tmin']) / 2

    if 'pr' in base_features and 'vap' in base_features:
        # Ratio Précipitation / Pression de vapeur (Indicateur de saturation)
        # Ajout de +1 pour éviter la division par zéro
        df_inter['pr_vap_ratio'] = df_inter['pr'] / (df_inter['vap'] + 1)

    if 'soil' in base_features and 'pr' in base_features:
        # Interaction Sol-Pluie (Boue/Humidité stagnante)
        df_inter['soil_pr_mult'] = df_inter['soil'] * df_inter['pr']

    # Création de variables polynomiales (carrés) pour les relations non-linéaires
    for col in base_features:
        if col in df.columns and pd.api.types.is_numeric_dtype(df[col]):
            df_inter[f'{col}_squared'] = df_inter[col] ** 2

    # Garder uniquement les nouvelles features + les anciennes
    new_cols = [c for c in df_inter.columns if c not in df.columns]
    print(f"{len(new_cols)} nouvelles variables d'interaction créées : {new_cols}")

    return df_inter

# ==========================================
# Exécution sur les jeux propres
# ==========================================

# Liste de vos 5 variables actuelles (à ajuster avec vos noms exacts)
# features_final correspond à la liste que vous avez utilisée pour le score de 0.81
X_train_inter = generate_safe_interactions(X_train_clean, features_final)
X_test_inter = generate_safe_interactions(X_test_clean, features_final)

# Relancez l'Ensemble (LightGBM + CatBoost) sur X_train_inter et X_test_inter

In [ ]:
# 1. Ratio Précipitation / Vapeur (Indicateur de saturation de l'air)
X_train_inter['ppt_vap_ratio'] = X_train_inter['ppt'] / (X_train_inter['vap'] + 1)
X_test_inter['ppt_vap_ratio'] = X_test_inter['ppt'] / (X_test_inter['vap'] + 1)

# 2. Stress Thermique et Hydrique (Sécheresse + Chaleur)
X_train_inter['drought_heat_stress'] = X_train_inter['pdsi'] * X_train_inter['tmax']
X_test_inter['drought_heat_stress'] = X_test_inter['pdsi'] * X_test_inter['tmax']

print(f"Nombre total de variables pour l'entraînement : {X_train_inter.shape[1]}")

In [ ]:
import catboost as cb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

# ==========================================
# 1. Réentraînement de l'Ensemble (11 variables)
# ==========================================
print("Entraînement de l'Ensemble sur les 11 variables d'interaction...")

# Identification automatique des colonnes catégorielles pour CatBoost
cat_features_idx = [i for i, col in enumerate(X_train_inter.columns) if X_train_inter[col].dtype.name == 'category']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_test_preds_inter = np.zeros(len(X_test_inter))
cb_test_preds_inter = np.zeros(len(X_test_inter))

for train_idx, val_idx in skf.split(X_train_inter, y_train_clean):
    X_tr, y_tr = X_train_inter.iloc[train_idx], y_train_clean.iloc[train_idx]
    X_va, y_va = X_train_inter.iloc[val_idx], y_train_clean.iloc[val_idx]

    # LightGBM
    lgb_model = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=31, is_unbalance=True, random_state=42, verbose=-1)
    lgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    lgb_test_preds_inter += lgb_model.predict_proba(X_test_inter)[:, 1] / 5

    # CatBoost (Correction : passage des cat_features)
    cb_model = cb.CatBoostClassifier(learning_rate=0.05, depth=6, auto_class_weights='Balanced', random_seed=42, verbose=0)
    cb_model.fit(X_tr, y_tr, cat_features=cat_features_idx, eval_set=(X_va, y_va), early_stopping_rounds=50)
    cb_test_preds_inter += cb_model.predict_proba(X_test_inter)[:, 1] / 5

final_ensemble_probs_inter = (lgb_test_preds_inter + cb_test_preds_inter) / 2

# ==========================================
# 2. Pseudo-Étiquetage Assoupli
# ==========================================
print("\nDémarrage du Pseudo-Étiquetage avec les nouvelles probabilités...")

seuil_presence = 0.85
seuil_absence = 0.15

X_test_pseudo = X_test_inter.copy()
X_test_pseudo['prob_ensemble'] = final_ensemble_probs_inter

pseudo_positives = X_test_pseudo[X_test_pseudo['prob_ensemble'] >= seuil_presence].copy()
pseudo_positives['target'] = 1

pseudo_negatives = X_test_pseudo[X_test_pseudo['prob_ensemble'] <= seuil_absence].copy()
pseudo_negatives['target'] = 0

pseudo_positives = pseudo_positives.drop(columns=['prob_ensemble'])
pseudo_negatives = pseudo_negatives.drop(columns=['prob_ensemble'])

print(f"Nouvelles observations 'Présence' (confiance > 85%) : {len(pseudo_positives)}")
print(f"Nouvelles observations 'Absence' (confiance < 15%) : {len(pseudo_negatives)}")

# Préparation du nouveau jeu d'entraînement boosté pour la phase finale
df_train_temp = X_train_inter.copy()
df_train_temp['target'] = y_train_clean
df_train_boosted = pd.concat([df_train_temp, pseudo_positives, pseudo_negatives], ignore_index=True)

print(f"Taille du jeu d'entraînement boosté prêt pour l'assaut final : {len(df_train_boosted)}")

In [ ]:
import catboost as cb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

# 1. Préparation du jeu final
X_final = df_train_boosted.drop(columns=['target'])
y_final = df_train_boosted['target']

# Identification des catégories
cat_features_final = [i for i, col in enumerate(X_final.columns) if X_final[col].dtype.name == 'category']

print(f"Entraînement final sur {len(X_final)} observations...")

# 2. Entraînement de l'Ensemble Final
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_final_preds = np.zeros(len(X_test_inter))
cb_final_preds = np.zeros(len(X_test_inter))

for train_idx, val_idx in skf.split(X_final, y_final):
    X_tr, y_tr = X_final.iloc[train_idx], y_final.iloc[train_idx]
    X_va, y_va = X_final.iloc[val_idx], y_final.iloc[val_idx]

    # LightGBM
    lgb_final = lgb.LGBMClassifier(learning_rate=0.03, num_leaves=31, is_unbalance=True, random_state=42, verbose=-1)
    lgb_final.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(100, verbose=False)])
    lgb_final_preds += lgb_final.predict_proba(X_test_inter)[:, 1] / 5

    # CatBoost
    cb_final = cb.CatBoostClassifier(learning_rate=0.03, depth=6, auto_class_weights='Balanced', random_seed=42, verbose=0)
    cb_final.fit(X_tr, y_tr, cat_features=cat_features_final, eval_set=(X_va, y_va), early_stopping_rounds=100)
    cb_final_preds += cb_final.predict_proba(X_test_inter)[:, 1] / 5

# 3. Probabilités combinées
final_probs = (lgb_final_preds + cb_final_preds) / 2

# 4. Génération de la soumission (Seuil optimisé)
# Note : On reste sur un seuil de 0.5 car les modèles sont déjà équilibrés via is_unbalance/auto_class_weights
final_classes = (final_probs > 0.5).astype(int)

submission_boosted = pd.DataFrame({
    'ID': test_ready_enc['ID'],
    'Occurrence Status': final_classes
})

filename = 'SUBMISSION_BOOSTED_FINAL.csv'
submission_boosted.to_csv(filename, index=False)

print(f"✅ Soumission finale générée : {filename}")
print("Distribution des classes :")
print(submission_boosted['Occurrence Status'].value_counts(normalize=True) * 100)